In [1]:
import torch
print(torch.__version__)
print("GPU available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

2.8.0+cpu
GPU available: False
Using device: cpu


### Tensors

In [7]:
import torch

# Create
x = torch.tensor([[1., 2.], [3., 4.]])          # from Python lists
z = torch.zeros((2,3))                          # zeros
r = torch.randn(3, 4)                           # normal random

# Shape & dtype
print("x,", x, x.shape, x.dtype)

# Basic ops (broadcasting works like NumPy)
y = torch.ones_like(x)
print("y,", y, x.shape, x.dtype)
s = x + y
print("x+y", s, s.shape, s.dtype)
m = x @ x.T
print("x+y", s, s.shape, s.dtype)                               # matrix multiply
m

x, tensor([[1., 2.],
        [3., 4.]]) torch.Size([2, 2]) torch.float32
y, tensor([[1., 1.],
        [1., 1.]]) torch.Size([2, 2]) torch.float32
x+y tensor([[2., 3.],
        [4., 5.]]) torch.Size([2, 2]) torch.float32
x+y tensor([[2., 3.],
        [4., 5.]]) torch.Size([2, 2]) torch.float32


tensor([[ 5., 11.],
        [11., 25.]])

In [8]:
a = torch.randn(2, 3)
b = a.numpy()                    # tensor -> numpy (shares memory!)
c = torch.from_numpy(b)          # numpy -> tensor

## Autograd (automatic differentiation)

PyTorch records ops on tensors that have `requires_grad=True`, then computes gradients with `.backward()`.

The gradients `∂loss/∂w` and `∂loss/∂b` represent how much the loss would change with respect to small changes in the parameters `w` and `b`.<br>
These gradients are computed through the computational graph that was built during the forward pass.<br>

Here's what's actually happening:

**Forward pass**: `y_pred = (w * x).sum() + b` creates a computational graph<br>
**Loss computation**: `loss = (y_pred - y_true)**2`<br>
**Backward pass**: `loss.backward()` computes gradients by applying the chain rule through the computational graph<br>
<br>
The value of `x` influences the gradients because it's part of the forward computation, but the gradients themselves are derivatives of the loss with respect to the parameters `w` and `b`.

More precisely:<br>
`w.grad` contains `∂loss/∂w = 2(y_pred - y_true) * x`<br>
`b.grad` contains `∂loss/∂b = 2(y_pred - y_true)`

In [ ]:
w = torch.randn(3, requires_grad=True)
b = torch.randn(1, requires_grad=True)

x = torch.tensor([2.0, -1.0, 0.5])
y_true = torch.tensor(1.0)

y_pred = (w * x).sum() + b
loss = (y_pred - y_true)**2

loss.backward()   # computes ∂loss/∂w=w.grad and ∂loss/∂b=b.grad
print(w.grad, b.grad) # Evaluates at x


tensor([ 1.3542, -0.6771,  0.3386]) tensor([0.6771])


The gradients are computed using the **current values** of all variables that participate in the computational graph - in this case:

- The current values of parameters `w` and `b`
- The current values of input `x`
- The current value of target `y_true`

When `loss.backward()` is called, it computes:

`∂loss/∂w` using the current values of `w`, `x`, `y_true`, and `b`<br>
`∂loss/∂b` using the current values of `w`, `x`, `y_true`, and `b`<br>

So the comment "*Evaluates at x*" could be more precisely written as "*Gradients computed at current values of all variables*" or "*Gradients evaluated at the current point in parameter space*".

This is why in training loops, you:

1. Compute gradients with current parameter values
2. Update parameters using those gradients
3. Zero gradients before the next iteration
4. Repeat with the new parameter values

Each gradient computation gives you the direction to move from your current position in the parameter landscape.

Disable grad when you’re just evaluating/inferencing:

In [20]:
with torch.no_grad():
    y = (w * x).sum() + b